# Week 33, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w033d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def half(n: int) -> int:
    return n / 2

print(half(7), half.__annotations__["return"].__name__)

**Predict before you run:** what will this print?



In [ ]:
stock = {"rice": 4, "salt": 0}
for item in ["rice", "salt", "yam"]:
    qty = stock.get(item)
    print(item, qty, "missing" if qty is None else "known")

**Predict before you run:** what will this print?



In [ ]:
import re

m = re.match(r"([A-Z]+)\d", "ls1 4ap")
print(m, m is None)
m = re.match(r"([A-Z]+)\d", "ls1 4ap".upper())
print(m.group(1))

**Predict before you run:** what will this print?



In [ ]:
class Box[T]:
    def __init__(self, item: T) -> None:
        self.item = item

b = Box[int]("not a number")
print(b.item, type(b).__name__, len(Box.__type_params__))

**Predict before you run:** what will this print?



In [ ]:
from typing import cast, get_origin, get_args

print(cast(list[int], "abc"), get_origin(dict[str, int]).__name__, get_args(tuple[int, ...]))

### Exercise: Hints for a small module

Add hints to every function, without changing behaviour. Import `Iterable` from `collections.abc`. `names` is any iterable of strings; `index_of` returns a position or `None`; `totals` maps names to lists of whole-number scores and returns a dictionary of totals; `pairs` returns a list of (name, total) tuples.

In [ ]:
from collections.abc import Iterable

def clean(names):
    return [n.strip().title() for n in names if n.strip()]

def index_of(names, wanted):
    for i, n in enumerate(names):
        if n == wanted:
            return i
    return None

def totals(scores):
    return {name: sum(marks) for name, marks in scores.items()}

def pairs(scores):
    return sorted(totals(scores).items())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"the hints\"\n    from typing import get_type_hints as h\n    from collections.abc import Iterable as It\n    assert h(clean) == {\"names\": It[str], \"return\": list[str]}, f\"clean: {h(clean)}\"\n    assert h(index_of) == {\"names\": list[str], \"wanted\": str, \"return\": int | None}, f\"index_of: {h(index_of)}\"\n    assert h(totals) == {\"scores\": dict[str, list[int]], \"return\": dict[str, int]}, f\"totals: {h(totals)}\"\n    assert h(pairs) == {\"scores\": dict[str, list[int]], \"return\": list[tuple[str, int]]}, f\"pairs: {h(pairs)}\"\n    assert clean([\" ada \", \"\", \"BO\"]) == [\"Ada\", \"Bo\"] and index_of([\"a\", \"b\"], \"b\") == 1 and index_of([], \"x\") is None\n    assert pairs({\"Bo\": [1, 2], \"Ada\": [5]}) == [(\"Ada\", 5), (\"Bo\", 3)]")

### Exercise: A generic group_by

Write `group_by[T, K](items: Iterable[T], key: Callable[[T], K]) -> dict[K, list[T]]`, which groups items by the result of `key`, keeping the items' order within each group and the groups in the order their keys first appear. Import `Callable` and `Iterable` from `collections.abc`.

In [ ]:
from collections.abc import Callable, Iterable

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_group():\n    \"grouping\"\n    from typing import get_type_hints as h\n    from collections.abc import Callable as C, Iterable as It\n    T, K = group_by.__type_params__\n    assert h(group_by) == {\"items\": It[T], \"key\": C[[T], K], \"return\": dict[K, list[T]]}, f\"hints: {h(group_by)}\"\n    assert group_by([\"fig\", \"yam\", \"plantain\", \"okra\", \"kiwi\"], len) == {3: [\"fig\", \"yam\"], 8: [\"plantain\"], 4: [\"okra\", \"kiwi\"]}\n    assert group_by(range(6), lambda n: n % 2 == 0) == {True: [0, 2, 4], False: [1, 3, 5]} and group_by([], str) == {}")

### Exercise: Forge: a typed stock keeper

Write `Product`, `money`, `Repository`, `load_products` and `Inventory` to the specification above.

In [ ]:
import json
import re
from collections.abc import Callable, Iterable
from dataclasses import dataclass
from typing import Any

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("DATA = '''[\n  {\"sku\": \"RIC-001\", \"name\": \"Rice\", \"pence\": 450, \"tags\": [\"grain\", \"staple\"]},\n  {\"sku\": \"OIL-002\", \"name\": \"Palm oil\", \"pence\": 900, \"tags\": [\"staple\"]},\n  {\"sku\": \"PEP-003\", \"name\": \"Pepper\", \"pence\": 120}\n]'''\ndef test_hints():\n    \"every hint\"\n    from typing import get_type_hints as h\n    from collections.abc import Callable as C, Iterable as It\n    P, R, I = ns[\"Product\"], ns[\"Repository\"], ns[\"Inventory\"]\n    assert h(P) == {\"sku\": str, \"name\": str, \"pence\": int, \"tags\": tuple[str, ...]}, f\"Product: {h(P)}\"\n    assert h(money) == {\"pence\": int, \"return\": str} and h(load_products) == {\"text\": str, \"return\": list[P]}\n    K, V = R.__type_params__\n    assert h(R.__init__) == {\"key\": C[[V], K], \"return\": type(None)} and h(R.add) == {\"item\": V, \"return\": type(None)}\n    assert h(R.get) == {\"key\": K, \"return\": V | None} and h(R.find) == {\"test\": C[[V], bool], \"return\": list[V]}\n    assert h(R.all) == {\"return\": list[V]} and h(R.__len__) == {\"return\": int}\n    assert h(I.__init__) == {\"products\": It[P], \"return\": type(None)}\n    assert h(I.restock) == {\"sku\": str, \"qty\": int, \"return\": int} and h(I.sell) == {\"sku\": str, \"qty\": int, \"return\": int}\n    assert h(I.value) == {\"return\": int} and h(I.low_stock) == {\"limit\": int, \"return\": list[str]}\n    assert h(I.tagged) == {\"tag\": str, \"return\": list[P]} and h(I.price_list) == {\"fmt\": C[[int], str], \"return\": list[str]}\ndef test_repository():\n    \"the repository\"\n    R = ns[\"Repository\"]\n    r = R(len)\n    r.add(\"fig\")\n    r.add(\"plantain\")\n    assert r.get(3) == \"fig\" and r.get(5) is None and len(r) == 2 and r.all() == [\"fig\", \"plantain\"] and r.find(lambda s: \"a\" in s) == [\"plantain\"]\n    try:\n        r.add(\"yam\")\n        assert False, \"a duplicate key should raise ValueError\"\n    except ValueError:\n        pass\ndef test_load():\n    \"loading\"\n    P = ns[\"Product\"]\n    ps = load_products(DATA)\n    assert ps[0] == P(\"RIC-001\", \"Rice\", 450, (\"grain\", \"staple\")) and ps[2].tags == () and len(ps) == 3\n    for bad, start in [('{\"sku\": \"x\"}', \"products must be a list\"), ('[{\"sku\": \"RIC-1\", \"name\": \"Rice\", \"pence\": 1}]', \"product 0:\"),\n                       ('[{\"sku\": \"RIC-001\", \"name\": \"Rice\", \"pence\": 1}, {\"sku\": \"OIL-002\", \"name\": \"\", \"pence\": 1}]', \"product 1:\"),\n                       ('[{\"sku\": \"RIC-001\", \"name\": \"Rice\", \"pence\": true}]', \"product 0:\"), ('[{\"sku\": \"RIC-001\", \"name\": \"Rice\", \"pence\": -1}]', \"product 0:\"),\n                       ('[{\"sku\": \"RIC-001\", \"name\": \"Rice\", \"pence\": 1, \"tags\": [\"a\", 2]}]', \"product 0:\"),\n                       ('[{\"sku\": \"RIC-001\", \"name\": \"Rice\", \"pence\": 1, \"colour\": \"white\"}]', \"product 0:\"),\n                       ('[{\"sku\": \"RIC-001\", \"name\": \"Rice\"}]', \"product 0:\"), ('[3]', \"product 0:\")]:\n        try:\n            load_products(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError as err:\n            assert str(err).startswith(start), f\"message should start {start!r}: {err}\"\n    try:\n        load_products(\"[not json\")\n        assert False, \"bad JSON should raise ValueError\"\n    except ValueError:\n        pass\ndef test_inventory():\n    \"the inventory\"\n    I = ns[\"Inventory\"]\n    inv = I(load_products(DATA))\n    assert inv.restock(\"RIC-001\", 10) == 10 and inv.restock(\"OIL-002\", 3) == 3 and inv.sell(\"RIC-001\", 4) == 6\n    assert inv.value() == 6 * 450 + 3 * 900 and inv.low_stock() == [\"OIL-002\", \"PEP-003\"] and inv.low_stock(7) == [\"OIL-002\", \"PEP-003\", \"RIC-001\"]\n    assert [p.sku for p in inv.tagged(\"staple\")] == [\"RIC-001\", \"OIL-002\"] and inv.tagged(\"frozen\") == []\n    assert inv.price_list() == [\"RIC-001 Rice \u00a34.50\", \"OIL-002 Palm oil \u00a39.00\", \"PEP-003 Pepper \u00a31.20\"]\n    assert inv.price_list(lambda p: f\"{p}p\")[2] == \"PEP-003 Pepper 120p\" and money(-305) == \"-\u00a33.05\"\n    for f, err in [(lambda: inv.sell(\"OIL-002\", 4), ValueError), (lambda: inv.restock(\"XXX-999\", 1), KeyError),\n                   (lambda: inv.sell(\"RIC-001\", 0), ValueError), (lambda: I(load_products(DATA) * 2), ValueError)]:\n        try:\n            f()\n            assert False, f\"should raise {err.__name__}\"\n        except err:\n            pass\n    assert inv.sell(\"OIL-002\", 3) == 0 and inv.value() == 2700")

> **Hints as a design tool.** Notice how much the hints said before you wrote a line of the bodies. `Repository[K, V]` with `key: Callable[[V], K]` told you the whole design: items in, keys worked out from them. `get(...) -> V | None` told every caller to deal with a missing item; `load_products(text: str) -> list[Product]` drew the line between untrusted text and trusted objects. Writing signatures first, and filling in bodies after, is a good way to design: if a signature is hard to write, the function probably does too much or too little. Next week turns that kind of thinking into principles.

### Exercise: Forge bonus: typed signals

Write a generic class `Signal[T]`, fully annotated:

- `__init__(self) -> None`.
- `connect(self, handler: Callable[[T], None]) -> Callable[[], None]`: adds a listener, and returns a **function** that, when called, disconnects that listener (calling it again does nothing).
- `emit(self, value: T) -> int`: calls every connected listener with the value, in the order connected, and returns how many were called.
- `__len__(self) -> int`: how many listeners are connected.
- A generic method `map[R](self, f: Callable[[T], R]) -> "Signal[R]"` (the return hint in quotes, because the class isn't finished while its body is being read): returns a **new** signal that emits `f(value)` whenever this one emits `value`.
- A method `filter(self, test: Callable[[T], bool]) -> "Signal[T]"`: returns a new signal that emits only the values for which `test` is True.

Connecting the same function twice connects it twice. A listener that disconnects itself (or others) while `emit` is running must not cause any listener to be skipped or called twice during that emit.

In [ ]:
from collections.abc import Callable

class Signal[T]:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"the hints\"\n    from typing import get_type_hints as h\n    from collections.abc import Callable as C\n    S = ns[\"Signal\"]\n    T, = S.__type_params__\n    assert h(S.connect) == {\"handler\": C[[T], None], \"return\": C[[], None]} and h(S.emit) == {\"value\": T, \"return\": int}\n    assert h(S.__len__) == {\"return\": int} and h(S.__init__) == {\"return\": type(None)}\n    R, = S.map.__type_params__\n    m, f = S.map.__annotations__, S.filter.__annotations__\n    assert m[\"f\"] == C[[T], R] and m[\"return\"] in (\"Signal[R]\", S[R]), f\"map: {m}\"\n    assert f[\"test\"] == C[[T], bool] and f[\"return\"] in (\"Signal[T]\", S[T]), f\"filter: {f}\"\ndef test_signals():\n    \"signals\"\n    S = ns[\"Signal\"]\n    got = []\n    paid = S()\n    off = paid.connect(lambda p: got.append((\"a\", p)))\n    paid.connect(lambda p: got.append((\"b\", p)))\n    assert paid.emit(500) == 2 and got == [(\"a\", 500), (\"b\", 500)] and len(paid) == 2\n    off()\n    off()\n    assert paid.emit(7) == 1 and got[-1] == (\"b\", 7) and len(paid) == 1\n    doubled = paid.map(lambda p: p * 2)\n    big = paid.filter(lambda p: p >= 1000)\n    seen = []\n    doubled.connect(seen.append)\n    big.connect(lambda p: seen.append((\"big\", p)))\n    paid.emit(600)\n    paid.emit(1500)\n    assert seen == [1200, 3000, (\"big\", 1500)], f\"seen {seen}\"\ndef test_self_disconnect():\n    \"disconnecting during emit\"\n    S = ns[\"Signal\"]\n    s = S()\n    calls = []\n    def once(v):\n        calls.append((\"once\", v))\n        stop()\n    stop = s.connect(once)\n    s.connect(lambda v: calls.append((\"always\", v)))\n    s.emit(1)\n    s.emit(2)\n    assert calls == [(\"once\", 1), (\"always\", 1), (\"always\", 2)], f\"calls {calls}\"\n    h = lambda v: calls.append((\"twice\", v))\n    s.connect(h)\n    s.connect(h)\n    assert s.emit(3) == 3")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [the basics](w033d1.html), [collections and Optional](w033d2.html), [running mypy](w033d3.html), [generics](w033d4.html) or [static and runtime types](w033d5.html).

## Mixed review

## Forge: a typed stock keeper

A small shop keeps its products in a JSON file and wants a program that tracks stock. The program is to be written **fully typed**: every function and method has hints on every parameter (except `self`) and on its return value, exactly as given below. Data from the file is checked once, when it's loaded; after that, the hints are trusted.

**`Product`**, a frozen dataclass with `sku: str`, `name: str`, `pence: int` and `tags: tuple[str, ...] = ()`.

**`money(pence: int) -> str`** gives `£4.50` (and `-£3.05` for negative amounts).

**`Repository[K, V]`**, a generic class that stores items by a key worked out from each item.

- `__init__(self, key: Callable[[V], K]) -> None`.
- `add(self, item: V) -> None`: raises `ValueError` if an item with the same key is already there.
- `get(self, key: K) -> V | None`.
- `find(self, test: Callable[[V], bool]) -> list[V]`: the items for which `test` is True, in the order added.
- `all(self) -> list[V]`, in the order added, and `__len__(self) -> int`.

**`load_products(text: str) -> list[Product]`** reads JSON text that must be a list of objects. Each object has exactly the keys `sku` (three capital letters, a hyphen and three digits, like `RIC-001`), `name` (a non-empty string), `pence` (a whole number of 0 or more, not a `bool`), and optionally `tags` (a list of strings). Any problem raises `ValueError` with a message starting `product 2:` (counting from 0) for a problem with one product, or `products must be a list` if the top level isn't a list. Invalid JSON raises `ValueError` too.

**`Inventory`**, built on a `Repository[str, Product]` keyed by SKU, and a stock count for each product, starting at 0.

- `__init__(self, products: Iterable[Product]) -> None`: a duplicate SKU raises `ValueError`.
- `restock(self, sku: str, qty: int) -> int` and `sell(self, sku: str, qty: int) -> int` return the new stock level. An unknown SKU raises `KeyError`, a `qty` below 1 raises `ValueError`, and selling more than is in stock raises `ValueError` and changes nothing.
- `value(self) -> int`: the total price of everything in stock.
- `low_stock(self, limit: int = 5) -> list[str]`: the SKUs with fewer than `limit` in stock, sorted.
- `tagged(self, tag: str) -> list[Product]`: the products with that tag, in the order loaded.
- `price_list(self, fmt: Callable[[int], str] = money) -> list[str]`: one line per product, in the order loaded, like `RIC-001 Rice £4.50`, with the price formatted by `fmt`.

## Forge bonus: typed signals

A **signal** lets one part of a program announce events to any number of others without knowing who they are: "a payment arrived", "a parcel was scanned". Each listener is a function. Typed generically, a `Signal[T]` only accepts listeners that take a `T`, and only emits `T`s.

## Weekly quiz

## Week 33 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can annotate functions, variables and classes, read hints with `get_type_hints`, and use `Self`. ([Day 1](w033d1.html))
- I can write hints for collections, values that might be `None` and unions, and narrow them. ([Day 2](w033d2.html))
- I can run mypy, read its reports, configure its strictness, and silence an error properly. ([Day 3](w033d3.html))
- I can write generic functions and classes, with bounds and constraints, and describe functions with `Callable`. ([Day 4](w033d4.html))
- I can explain static and runtime checking, check outside data at the boundary, and use `cast` correctly. ([Day 5](w033d5.html))
- I built a fully typed stock keeper and typed signals. (Today)

Tomorrow is a **rest day**. Next week: **design principles**, starting with cohesion and coupling: how to split a program into parts that are easy to change.